In [ ]:
import xarray as xr
from pathlib import Path
from utils import plot_global_field
from config_utils import load_config
import torch
from model import SimpleModel
from preprocessing import create_2024_preprocessor_no_downscaling
from utils import dataset_from_array, ordered_dataset
from eval import compute_nn_kernel_autograd
import numpy as np
import pandas as pd
base = Path("/home/ezrah442/group/hanhuang/paper_related/4xco2_coupling_sw")


In [ ]:
ds_dR_01 = xr.open_dataset(base/"4xco2_dR_01.nc")
ds_dR_06 = xr.open_dataset(base/"4xco2_dR_06.nc")
ds_dalb = xr.open_dataset(base/"4xco2_dalb.nc")

output_path = Path("closure_test")
output_path.mkdir(exist_ok=True)

def plot_months(data_by_month, title, filename, *, lon, lat, vmin, vmax, label=r"$W/m^2$", **kwargs):
    for month, data in data_by_month.items():
        month_text = f"{month:02d}"
        plot_global_field(
            data,
            lon,
            lat,
            title.replace("{month:02d}", month_text).replace("{month}", str(month)),
            output_path / filename.replace("{month:02d}", month_text).replace("{month}", str(month)),
            vmin=vmin,
            vmax=vmax,
            label=label,
            **kwargs,
        )

def plot_ranges(data_by_month, title, filename, *, lon, lat, ranges, **kwargs):
    for suffix, vmin, vmax in ranges:
        plot_months(
            data_by_month,
            title,
            filename.replace("{suffix}", suffix),
            lon=lon,
            lat=lat,
            vmin=vmin,
            vmax=vmax,
            **kwargs,
        )


In [ ]:
plot_months(
    {month: ds_dalb.sel(month=month).dalb for month in (1, 6)},
    "dalb 2015-{month:02d}",
    "dalb_{month:02d}.png",
    lon=ds_dalb.longitude,
    lat=ds_dalb.latitude,
    vmin=-1,
    vmax=1,
    label=r"",
)
plot_months(
    {1: ds_dR_01.dTOA.sel(up_down_net=1), 6: ds_dR_06.dTOA.sel(up_down_net=1)},
    r"$\Delta R$ 2015-{month:02d}",
    "dR_{month:02d}_net.png",
    lon=ds_dR_01.longitude,
    lat=ds_dR_01.latitude,
    vmin=-400,
    vmax=400,
)

In [ ]:
config = load_config("configs/model/fal/sobolev_mar_june_sep_dec_2011_2014.yaml")
checkpoint_data = torch.load(
    Path(config.train.checkpoint_dir) / "best_model.pt",
    map_location="cpu",
    weights_only=False,
)

model = SimpleModel(checkpoint_data["config"])
model.load_state_dict(checkpoint_data["model_state_dict"])
model.eval()

preprocessor = create_2024_preprocessor_no_downscaling()
preprocessor.load(config.preprocess.params_dir)

dates = ["2015-01", "2015-06"]
ds_era5 = xr.open_dataset("data/era5/era5_single_levels_monthly_2015.nc").sel(date=dates)


In [ ]:
K_nn_01, lon, lat = compute_nn_kernel_autograd(ds_era5.sel(date=dates[0]), preprocessor, model, config)
K_nn_06, _, _ = compute_nn_kernel_autograd(ds_era5.sel(date=dates[1]), preprocessor, model, config)
K_nn = xr.Dataset(
    data_vars=dict(kernel=(["month", "latitude", "longitude"], [K_nn_01, K_nn_06])),
    coords=dict(longitude=lon, latitude=lat, month=[1, 6]),
)
dR_alb_nn = ds_dalb.dalb * K_nn.interp(
    latitude=ds_dalb.latitude,
    longitude=ds_dalb.longitude,
    kwargs={"fill_value": "extrapolate"},
).kernel * 100

plot_months(
    {1: K_nn_01, 6: K_nn_06},
    "NN albedo kernel 2015-{month:02d}",
    "2015-{month:02d}_nn_fal_autograd_kernel.png",
    lon=lon,
    lat=lat,
    vmin=-4,
    vmax=4,
    label=r"",
)
plot_ranges(
    {month: dR_alb_nn.sel(month=month) for month in (1, 6)},
    r"NN $\Delta R_{\text{fal}}$ 2015-{month:02d}",
    "dR_{month:02d}_fal_autograd_{suffix}nn.png",
    lon=dR_alb_nn.longitude,
    lat=dR_alb_nn.latitude,
    ranges=(("zoom_", -25, 25), ("", -300, 300)),
)


In [ ]:
preprocessed = preprocessor.transform(ds_era5)
lat = preprocessed.latitude
lon = preprocessed.longitude

def predict_tsr(ds: xr.Dataset, preprocessor, model) -> xr.DataArray:
    ds_ordered_np = (
        ordered_dataset(ds, "tsr")
        .to_dataarray()
        .transpose("date", "latitude", "longitude", "variable")
        .to_numpy()
    )
    data_torch = torch.from_numpy(ds_ordered_np).float()
    model_outputs = model(data_torch[:, :, :, : model.input_dim]).detach()
    return (
        preprocessor.preprocessors[-1]
        .inverse_transform(
            dataset_from_array(arr=model_outputs.numpy(), date=dates, lon=lon, lat=lat),
        )
        .to_dataarray()
        .squeeze(dim="variable", drop=True)
        .to_numpy()
        / (3600 * 24)
    )

def predict_delta_r_from_albedo(ds: xr.Dataset, dalb: xr.DataArray, preprocessor, model) -> xr.DataArray:
    ds_p = preprocessor.transform(ds)
    dalb = dalb.interp(latitude=ds.latitude, longitude=ds.longitude, kwargs={"fill_value": "extrapolate"})

    ds_copy = ds.copy(deep=True)
    ds_copy["fal"] = ds_copy["fal"] - dalb
    max_val = np.max(ds_copy.fal).values

    for date_value in dates:
        month = pd.to_datetime(date_value).month
        plot_global_field(
            ds.fal.sel(date=date_value).squeeze("date"),
            ds.fal.longitude,
            ds.fal.latitude,
            f"fal {date_value}",
            output_path / f"fal_{month:02d}.png",
            vmin=0,
            vmax=1,
            label=r"",
            cmap="viridis",
        )
        plot_global_field(
            ds_copy.fal.sel(date=date_value).squeeze("date"),
            ds.fal.longitude,
            ds.fal.latitude,
            f"perturbed fal {date_value}",
            output_path / f"fal_pert_{month:02d}.png",
            vmin=0,
            vmax=max_val,
            label=r"",
            cmap="viridis",
            annotation=f"{max_val:.2f}",
        )

    perturbed = preprocessor.transform(ds_copy)
    return predict_tsr(perturbed, preprocessor, model) - predict_tsr(ds_p, preprocessor, model)

dalb = ds_dalb.dalb.rename(month="date").assign_coords(date=[pd.to_datetime(d) for d in dates])
dR_nn = predict_delta_r_from_albedo(ds_era5, dalb, preprocessor, model)
dR_nn = xr.Dataset(
    data_vars=dict(dR=(["month", "latitude", "longitude"], dR_nn)),
    coords=dict(longitude=lon, latitude=lat, month=[1, 6]),
)

plot_ranges(
    {month: dR_nn.dR.sel(month=month) for month in (1, 6)},
    r"NN $\Delta R_{\text{fal}}$ 2015-{month:02d}",
    "dR_{month:02d}_fal_nn{suffix}.png",
    lon=preprocessed.longitude,
    lat=preprocessed.latitude,
    ranges=(("_zoom", -25, 25), ("", -300, 300)),
)


In [ ]:
def predict_delta_r_autograd_iterative(
    ds: xr.Dataset,
    dalb: xr.DataArray,
    preprocessor,
    model,
    config,
    step_size: float = 0.01,
) -> xr.DataArray:
    dalb = dalb.interp(
        latitude=ds.latitude,
        longitude=ds.longitude,
        kwargs={"fill_value": "extrapolate"},
    ).fillna(0)
    if "month" in dalb.dims:
        dalb = dalb.rename(month="date").assign_coords(date=ds.date)

    current = ds.copy(deep=True)
    remaining = dalb.copy()
    total = xr.zeros_like(dalb)
    n_steps = int(np.ceil(float(np.nanmax(np.abs(dalb))) / step_size))

    for i in range(n_steps):
        step = xr.where(
            np.abs(remaining) > step_size,
            np.sign(remaining) * step_size,
            remaining,
        )
        if not bool((np.abs(step) > 0).any()):
            break

        kernels = []
        for date_value in current.date.values:
            kernel, k_lon, k_lat = compute_nn_kernel_autograd(
                current.sel(date=[date_value]), preprocessor, model, config
            )
            kernels.append(
                xr.DataArray(
                    kernel,
                    dims=("latitude", "longitude"),
                    coords={"latitude": k_lat, "longitude": k_lon},
                )
            )

        kernel = xr.concat(kernels, dim=current.date).assign_coords(date=current.date)
        kernel = kernel.interp(
            latitude=step.latitude,
            longitude=step.longitude,
            kwargs={"fill_value": "extrapolate"},
        )
        total = total + kernel * (step / step_size)
        current["fal"] = current["fal"] + step.data
        remaining = remaining - step
        print(f"{i + 1}/{n_steps}")

    return total


dR_nn_iterative = predict_delta_r_autograd_iterative(ds_era5, ds_dalb.dalb, preprocessor, model, config, 0.01)


In [ ]:
plot_ranges(
    {
        month: dR_nn_iterative.sel(date=f"2015-{month:02d}").squeeze("date")
        for month in (1, 6)
    },
    r"NN iterative $\Delta R_{\text{fal}}$ 2015-{month:02d}",
    "dR_{month:02d}_fal_nn_it{suffix}.png",
    lon=dR_nn_iterative.longitude,
    lat=dR_nn_iterative.latitude,
    ranges=(("", -300, 300), ("_zoom", -25, 25)),
)


In [ ]:
plot_ranges(
    {
        1: ds_dR_01.dTOA_a.sel(up_down_net=1),
        6: ds_dR_06.dTOA_a.sel(up_down_net=1),
    },
    r"RRTM $\Delta R_{\text{fal}}$ 2015-{month:02d}",
    "dR_{month:02d}_fal_rrtm{suffix}.png",
    lon=ds_dR_01.longitude,
    lat=ds_dR_01.latitude,
    ranges=(("", -300, 300), ("_zoom", -25, 25)),
)


In [ ]:
rrtm_by_month = {
    1: ds_dR_01.dTOA_a.sel(up_down_net=1),
    6: ds_dR_06.dTOA_a.sel(up_down_net=1),
}
for name, values in {
    "autograd": dR_alb_nn,
    "nn": dR_nn.dR,
}.items():
    plot_months(
        {
            month: values.sel(month=month).interp(
                latitude=ds_dR_01.latitude,
                longitude=ds_dR_01.longitude,
                kwargs={"fill_value": "extrapolate"},
            ) - rrtm_by_month[month]
            for month in (1, 6)
        },
        r"NN-RRTM $\Delta R_{\text{fal}}$ 2015-{month:02d}",
        f"dR_{{month:02d}}_fal_{name}-rrtm.png",
        lon=ds_dR_01.longitude,
        lat=ds_dR_01.latitude,
        vmin=-50,
        vmax=50,
    )


In [ ]:
def plot_residuals(residual_by_month, title, filename):
    plot_months(
        {month: residual.sel(up_down_net=1) for month, residual in residual_by_month.items()},
        title,
        filename,
        lon=ds_dR_01.longitude,
        lat=ds_dR_01.latitude,
        vmin=-400,
        vmax=400,
    )

plot_residuals(
    {
        month: ds.dTOA - ds.dTOA_c - ds.dTOA_a - ds.dTOA_q
        for month, ds in ((6, ds_dR_06), (1, ds_dR_01))
    },
    "kernel residual 2015-{month:02d}",
    "dR_{month:02d}_kernel_residual.png",
)


In [ ]:
plot_residuals(
    {
        month: ds.dTOA - ds.dTOA_c - dR_alb_nn.sel(month=month).interp(
            latitude=ds.latitude,
            longitude=ds.longitude,
            kwargs={"fill_value": "extrapolate"},
        ) - ds.dTOA_q
        for month, ds in ((6, ds_dR_06), (1, ds_dR_01))
    },
    "kernel (with fal nn) residual 2015-{month:02d}",
    "dR_{month:02d}_nn_residual.png",
)
